# Lec8 — Diffusion Models
### Forward noising + learned reverse denoising

**Prerequisites:** probability basics (Gaussians), Lec2–Lec5 materials

| Part | Topic | Key idea |
|---|---|---|
| 1 | Distributions | image generation = learn a data distribution $p_{\text{data}}(x)$ |
| 2 | Diffusion view | generate by *transitioning distributions* from noise → data |
| 3 | Forward process | use a **Gaussian kernel transition** to progressively add noise |


---
## (1) Distribution modeling for image generation

In generative modeling, we want to learn a model distribution $p_\theta(x)$ that matches the (unknown) data distribution $p_{\text{data}}(x)$.

- If we can sample $x \sim p_\theta(x)$, we can generate images.
- The challenge: images are high-dimensional (e.g. $x \in \mathbb{R}^{784}$ for MNIST, or much larger for real images).

**Diffusion model idea (high level):**

Instead of generating $x$ in one shot, create a **sequence of intermediate distributions**

$$q_0(x) \to q_1(x) \to \cdots \to q_T(x)$$

such that:

- $q_0(x)$ is the data distribution (clean images)
- $q_T(x)$ is a simple distribution (e.g. standard Gaussian noise)

Then we learn a reverse process that goes from noise back to data.

You can think of each time step $t$ as defining a *noise level* and therefore an **intermediate distribution** $q_t(x_t)$.


---
## (2) Forward diffusion as a Gaussian-kernel transition

We define a **forward noising process** (a Markov chain):

$$q(x_t \mid x_{t-1}) = \mathcal{N}\big(\sqrt{1-\beta_t}\,x_{t-1},\; \beta_t I\big), \qquad t=1,\dots,T$$

Equivalently, we can sample it as:

$$x_t = \sqrt{1-\beta_t}\,x_{t-1} + \sqrt{\beta_t}\,\varepsilon, \qquad \varepsilon \sim \mathcal{N}(0,I).$$

Why this is a **Gaussian kernel** transition:

- Each step adds Gaussian noise, which is a **convolution with a Gaussian kernel** in the space of images.
- Repeating this transition pushes any complicated distribution toward a Gaussian (for large enough $T$).

A useful closed-form relationship (we’ll verify in code):

$$q(x_t \mid x_0) = \mathcal{N}\big(\sqrt{\bar\alpha_t}\,x_0,\; (1-\bar\alpha_t) I\big), \quad \bar\alpha_t = \prod_{s=1}^t (1-\beta_s).$$


In [25]:
import numpy as np
import torch
import matplotlib.pyplot as plt

from sklearn.datasets import fetch_openml
from torch.utils.data import TensorDataset, DataLoader

torch.manual_seed(42)
device = "cuda" if torch.cuda.is_available() else "cpu"
print('device:', device)


device: cpu


In [26]:
# Load MNIST (same approach as earlier lectures)
print("Loading MNIST... (cached after first run)")
mnist   = fetch_openml("mnist_784", version=1, as_frame=False, parser="auto")
X_mnist = mnist.data.astype(np.float32) / 255.0
y_mnist = mnist.target.astype(int)

X_tr, X_te = X_mnist[:60000], X_mnist[60000:]
y_tr, y_te = y_mnist[:60000], y_mnist[60000:]

def make_dl(X, y, batch_size, shuffle):
    Xt = torch.tensor(X).view(-1, 1, 28, 28)
    yt = torch.tensor(y)
    return DataLoader(TensorDataset(Xt, yt), batch_size=batch_size, shuffle=shuffle)

train_dl = make_dl(X_tr, y_tr, 256, True)
test_dl  = make_dl(X_te, y_te, 512, False)

imgs_test, labels_test = next(iter(test_dl))
print('batch:', imgs_test.shape)


Loading MNIST... (cached after first run)
batch: torch.Size([512, 1, 28, 28])


In [ ]:
# Forward process parameters

def linear_beta_schedule(T, beta_start=1e-4, beta_end=2e-2):
    return torch.linspace(beta_start, beta_end, T)

T = 200
betas  = linear_beta_schedule(T).to(device)        # betas[0] is beta_1
alphas = 1.0 - betas                              # alphas[0] is alpha_1
alpha_bar = torch.cumprod(alphas, dim=0)          # cumulative product of alphas

# For teaching clarity, define alpha_bar0[0] = 1 so that x_0 is exactly the clean image.
# alpha_bar0[t] stores the cumulative product up to step t (t=0..T).
alpha_bar0 = torch.cat([torch.ones(1, device=device), alpha_bar], dim=0)

print('T:', T)
print('beta range:', float(betas.min()), '->', float(betas.max()))
print('alpha_bar0[0]:', float(alpha_bar0[0]), 'alpha_bar0[T]:', float(alpha_bar0[T]))

# Closed-form sampling from q(x_t | x_0)

def q_sample(x0, t, noise=None):
    # Sample x_t given x_0 at discrete time index t (0..T)
    if noise is None:
        noise = torch.randn_like(x0)
    a_bar = alpha_bar0.gather(0, t).view(-1, 1, 1, 1)  # (B,1,1,1)
    return torch.sqrt(a_bar) * x0 + torch.sqrt(1.0 - a_bar) * noise


In [ ]:
# Visualise: forward noising of a single MNIST image across timesteps
x0 = imgs_test[0:1].to(device)

steps = [0, 1, 2, 5, 10, 20, 50, 100, 150, T]

def show_row(images, titles, figsize=(15, 2.2)):
    fig, axes = plt.subplots(1, len(images), figsize=figsize)
    for ax, img, ttl in zip(axes, images, titles):
        ax.imshow(img[0,0], cmap='gray', vmin=0, vmax=1)
        ax.axis('off')
        ax.set_title(ttl, fontsize=9)
    plt.tight_layout(); plt.show()

imgs = []
for t in steps:
    xt = q_sample(x0, torch.tensor([t], device=device))
    imgs.append(xt.detach().cpu().clamp(0, 1))

show_row(imgs, [f"t={t}" for t in steps])
print('Forward process: x_t gradually becomes pure noise as t increases.')


In [ ]:
# Visualise many images across timesteps (grid)
# Rows: different initial images x0
# Cols: the same image diffused to different time steps t
# Here we simulate the *Markov* forward chain so each step adds fresh Gaussian noise.

n_rows = 8
steps = [0, 1, 2, 5, 10, 20, 50, 100, 150, T]
max_t = max(steps)

x = imgs_test[:n_rows].to(device)
labels_batch = labels_test[:n_rows].cpu().numpy()

snap = {0: x.detach().cpu()}
for t in range(1, max_t + 1):
    eps = torch.randn_like(x)
    beta = betas[t - 1]
    x = torch.sqrt(1.0 - beta) * x + torch.sqrt(beta) * eps
    if t in steps:
        snap[t] = x.detach().cpu()

fig, axes = plt.subplots(n_rows, len(steps), figsize=(1.45*len(steps), 1.45*n_rows))

for r in range(n_rows):
    for c, t in enumerate(steps):
        ax = axes[r, c] if n_rows > 1 else axes[c]
        x_t = snap[t][r:r+1]
        ax.imshow(x_t[0,0].clamp(0, 1), cmap='gray', vmin=0, vmax=1)
        ax.axis('off')
        if r == 0:
            ax.set_title(f"t={t}", fontsize=9)
    axes[r, 0].set_ylabel(f"y={labels_batch[r]}", rotation=0, labelpad=18, fontsize=9, va='center')

plt.suptitle(
    f"Forward diffusion on multiple images (Markov chain)\n(each row = one x0 image; each column = a noise level up to t={T})",
    fontsize=11
)
plt.tight_layout(); plt.show()


---
## (3) Distribution transition (MNIST + toy 2D)

The forward process does not only corrupt one image — it transforms the **entire data distribution**.

A good way to see this is: for a fixed time step $t$, draw many samples from $q_t(x_t)$.

- At $t=0$, samples look like clean digits (data distribution).
- As $t$ increases, samples become progressively noisier.
- For large $t$, samples look like nearly pure Gaussian noise.

First we visualise this on MNIST by showing a *block of many samples* at each time step.
Then we repeat the same idea on a simple 2D mixture-of-Gaussians dataset.


In [ ]:
# Distribution snapshots on MNIST: samples from q_t(x_t)
# Each panel is a different time step t, showing many samples (a mini-grid) from that intermediate distribution.
# We simulate the Markov forward chain on a batch of many different x0 images.

steps = [0, 10, 50, 100, T]
max_t = max(steps)

n_rows, n_cols = 6, 6
n = n_rows * n_cols

# Pick n different clean images x0
perm = torch.randperm(imgs_test.shape[0])[:n]
x = imgs_test[perm].to(device)

snap = {0: x.detach().cpu()}
for t in range(1, max_t + 1):
    eps = torch.randn_like(x)
    beta = betas[t - 1]
    x = torch.sqrt(1.0 - beta) * x + torch.sqrt(beta) * eps
    if t in steps:
        snap[t] = x.detach().cpu()

blocks = []
for t in steps:
    x_t = snap[t].clamp(0, 1).numpy()
    canvas = np.zeros((28*n_rows, 28*n_cols))
    k = 0
    for i in range(n_rows):
        for j in range(n_cols):
            canvas[i*28:(i+1)*28, j*28:(j+1)*28] = x_t[k, 0]
            k += 1
    blocks.append(canvas)

fig, axes = plt.subplots(1, len(steps), figsize=(3.2*len(steps), 3.2))
for ax, t, canvas in zip(axes, steps, blocks):
    ax.imshow(canvas, cmap='gray', vmin=0, vmax=1)
    ax.axis('off')
    ax.set_title(f"t={t}", fontsize=10)

plt.suptitle('Intermediate distributions q_t(x_t): many chains (many x0 images) at each noise level', fontsize=12)
plt.tight_layout(); plt.show()


In [ ]:
# Toy 2D distribution: many chains -> distribution at each step
# Here, EACH POINT is one forward Markov chain in 2D.
# For each time step t, plotting all points gives a snapshot of the marginal q_t(x_t).

rng = np.random.default_rng(0)

def sample_mog(n=20000):
    centers = np.array(
        [[2.5, 0.0], [-2.5, 0.0], [0.0, 2.5], [0.0, -2.5]],
        dtype=np.float32,
    )
    comp = rng.integers(0, len(centers), size=n)
    X = centers[comp] + 0.35 * rng.normal(size=(n, 2)).astype(np.float32)
    return X

# Many initial points (many chains)
N = 25000
x = torch.tensor(sample_mog(N), device=device)

# Timesteps to visualise (distribution snapshots)
steps = [0, 10, 30, 60, 100, 150, T]
max_t = max(steps)

snapshots = {0: x.detach().cpu().numpy()}

# Forward Markov chain: x_t = sqrt(1-beta_t) x_{t-1} + sqrt(beta_t) eps_t
for t in range(1, max_t + 1):
    eps = torch.randn_like(x)
    beta = betas[t - 1]  # transition 1 uses betas[0]
    x = torch.sqrt(1.0 - beta) * x + torch.sqrt(beta) * eps
    if t in steps:
        snapshots[t] = x.detach().cpu().numpy()

# Plot snapshots as a grid (each panel = distribution at one time step)
lims = (-6.0, 6.0)

ncols = 4
nrows = int(np.ceil(len(steps) / ncols))
fig, axes = plt.subplots(nrows, ncols, figsize=(4.0*ncols, 3.6*nrows))
axes = np.array(axes).reshape(nrows, ncols)

for k, t in enumerate(steps):
    r, c = divmod(k, ncols)
    ax = axes[r, c]
    xt = snapshots[t]
    ax.scatter(xt[:,0], xt[:,1], s=1.5, alpha=0.12)

    a = float(alpha_bar0[t].detach().cpu())
    ax.set_title(f"t={t}   (alpha_bar={a:.3f})")
    ax.set_aspect('equal', 'box')
    ax.set_xlim(*lims); ax.set_ylim(*lims)
    ax.set_xticks([]); ax.set_yticks([])

# Hide any empty panels
for k in range(len(steps), nrows*ncols):
    r, c = divmod(k, ncols)
    axes[r, c].axis('off')

plt.suptitle(
    "Forward Gaussian-kernel transition on a multi-modal distribution\\n"
    "(many chains: multi-modal MoG gradually becomes approximately single Gaussian)",
    fontsize=12,
)
plt.tight_layout(); plt.show()

# Optional: overlay a few trajectories to emphasize 'many chains'
show_trajectories = False
if show_trajectories:
    n_traj = 40
    x_traj = torch.tensor(sample_mog(n_traj), device=device)
    traj = [x_traj.detach().cpu().numpy()]
    for t in range(1, max_t + 1):
        eps = torch.randn_like(x_traj)
        beta = betas[t - 1]
        x_traj = torch.sqrt(1.0 - beta) * x_traj + torch.sqrt(beta) * eps
        if t in steps:
            traj.append(x_traj.detach().cpu().numpy())

    plt.figure(figsize=(6, 6))
    for i in range(n_traj):
        pts = np.stack([tr[i] for tr in traj], axis=0)
        plt.plot(pts[:,0], pts[:,1], alpha=0.35, linewidth=1)
        plt.scatter(pts[-1,0], pts[-1,1], s=12)
    plt.xlim(*lims); plt.ylim(*lims)
    plt.gca().set_aspect('equal', 'box')
    plt.title('Example forward trajectories (subset of chains)')
    plt.xticks([]); plt.yticks([])
    plt.tight_layout(); plt.show()


---
## (4) Reverse denoising, attempt 1: predict $x_{t-1}$ directly

A very natural first idea is to train a neural network that maps

$$f_\theta(x_t, t) \approx x_{t-1}.$$

**How to create supervision:** sample $x_0$ from the dataset, pick a time step $t$, use the forward process to generate a pair $(x_{t-1}, x_t)$, then train $f_\theta$ with MSE.

We will use the convention that $t=0$ is the clean image $x_0$, and $t=T$ is (almost) pure Gaussian noise.


In [ ]:
# Naive reverse model: predict x_{t-1} directly from (x_t, t)
# This is NOT the standard DDPM objective — we do it first for intuition.

import torch.nn as nn
import torch.nn.functional as F

def extract(a, t, x_shape):
    out = a.gather(0, t)
    return out.view(-1, *([1] * (len(x_shape) - 1)))

def sinusoidal_time_embedding(t, dim=64):
    half = dim // 2
    freqs = torch.exp(
        torch.linspace(0, np.log(10000), half, device=t.device, dtype=torch.float32) * (-1)
    )
    args = t.float().unsqueeze(1) * freqs.unsqueeze(0)
    emb = torch.cat([torch.sin(args), torch.cos(args)], dim=1)
    if dim % 2 == 1:
        emb = torch.cat([emb, torch.zeros_like(emb[:, :1])], dim=1)
    return emb

class XPrevModel(nn.Module):
    # Predict x_{t-1} directly from x_t
    def __init__(self, time_dim=64):
        super().__init__()
        self.time_mlp = nn.Sequential(
            nn.Linear(time_dim, 128), nn.ReLU(),
            nn.Linear(128, 128),
        )
        self.to_time1 = nn.Linear(128, 32)
        self.to_time2 = nn.Linear(128, 64)

        self.conv1 = nn.Conv2d(1, 32, 3, padding=1)
        self.conv2 = nn.Conv2d(32, 64, 3, padding=1)
        self.conv3 = nn.Conv2d(64, 64, 3, padding=1)
        self.conv4 = nn.Conv2d(64, 32, 3, padding=1)
        self.conv5 = nn.Conv2d(32, 1, 3, padding=1)

    def forward(self, x, t):
        t_emb = sinusoidal_time_embedding(t, dim=64)
        t_h = self.time_mlp(t_emb)

        h = F.relu(self.conv1(x) + self.to_time1(t_h).view(-1, 32, 1, 1))
        h = F.relu(self.conv2(h) + self.to_time2(t_h).view(-1, 64, 1, 1))
        h = F.relu(self.conv3(h))
        h = F.relu(self.conv4(h))
        return self.conv5(h)


---
### Training pairs for direct $x_{t-1}$ prediction

We want to train on pairs sampled from the forward chain **without** simulating all $T$ steps every time.

For each minibatch:

1. Sample $t \sim \text{Uniform}\{1,\dots,T\}$. (We never need to predict $x_{-1}$.)
2. Sample $x_{t-1}$ from the closed form marginal:

$$x_{t-1} = \sqrt{\bar\alpha_{t-1}}\,x_0 + \sqrt{1-\bar\alpha_{t-1}}\,\varepsilon_1.$$

3. Take one forward Markov step to get $x_t$:

$$x_t = \sqrt{1-\beta_t}\,x_{t-1} + \sqrt{\beta_t}\,\varepsilon_2.$$

Then train $f_\theta(x_t,t)$ to match $x_{t-1}$.


In [ ]:
# Train f_theta(x_t, t) \approx x_{t-1}

xprev_model = XPrevModel().to(device)
opt = torch.optim.Adam(xprev_model.parameters(), lr=1e-3)

epochs = 2
max_batches_per_epoch = 200  # cap runtime for lecture demo

print('Training naive x_{t-1} predictor...')
for epoch in range(epochs):
    xprev_model.train()
    total = 0.0

    for batch_idx, (x0, _) in enumerate(train_dl):
        if batch_idx >= max_batches_per_epoch:
            break

        x0 = x0.to(device)
        b = x0.shape[0]

        # Sample t in {1,...,T}. Target is x_{t-1}.
        t = torch.randint(1, T + 1, (b,), device=device, dtype=torch.long)
        t_prev = t - 1

        # 1) Sample x_{t-1} from the marginal q(x_{t-1} | x0)
        eps_prev = torch.randn_like(x0)
        a_bar_prev = extract(alpha_bar0, t_prev, x0.shape)
        x_prev = torch.sqrt(a_bar_prev) * x0 + torch.sqrt(1.0 - a_bar_prev) * eps_prev

        # 2) One Markov step to get x_t from x_{t-1}
        eps_step = torch.randn_like(x0)
        beta_t  = extract(betas,  t - 1, x0.shape)
        alpha_t = extract(alphas, t - 1, x0.shape)
        x_t = torch.sqrt(alpha_t) * x_prev + torch.sqrt(beta_t) * eps_step

        # Predict x_{t-1}
        x_prev_pred = xprev_model(x_t, t)
        loss = F.mse_loss(x_prev_pred, x_prev)

        opt.zero_grad()
        loss.backward()
        opt.step()

        total += loss.item()

    denom = max(1, min(len(train_dl), max_batches_per_epoch))
    print(f"Epoch {epoch+1}/{epochs}  loss={total/denom:.5f}")


In [ ]:
# Visualize: one-step denoising quality and naive generation

@torch.no_grad()
def naive_step_demo(model, x0, t_val=120):
    model.eval()
    b = x0.shape[0]
    t = torch.full((b,), t_val, device=device, dtype=torch.long)
    t_prev = t - 1

    # Build a consistent pair (x_{t-1}, x_t) from x0
    eps_prev = torch.randn_like(x0)
    a_bar_prev = extract(alpha_bar0, t_prev, x0.shape)
    x_prev = torch.sqrt(a_bar_prev) * x0 + torch.sqrt(1.0 - a_bar_prev) * eps_prev

    eps_step = torch.randn_like(x0)
    beta_t  = extract(betas,  t - 1, x0.shape)
    alpha_t = extract(alphas, t - 1, x0.shape)
    x_t = torch.sqrt(alpha_t) * x_prev + torch.sqrt(beta_t) * eps_step

    x_prev_pred = model(x_t, t)
    return x_t.cpu(), x_prev.cpu(), x_prev_pred.cpu()

# Compare true x_{t-1} vs predicted, for a mid-range t
x0_demo = imgs_test[:8].to(device)
x_t, x_prev_true, x_prev_pred = naive_step_demo(xprev_model, x0_demo, t_val=min(120, T))

fig, axes = plt.subplots(8, 3, figsize=(6.2, 12))
for i in range(8):
    axes[i,0].imshow(x_t[i,0].clamp(0,1), cmap='gray', vmin=0, vmax=1)
    axes[i,1].imshow(x_prev_true[i,0].clamp(0,1), cmap='gray', vmin=0, vmax=1)
    axes[i,2].imshow(x_prev_pred[i,0].clamp(0,1), cmap='gray', vmin=0, vmax=1)
    for j in range(3):
        axes[i,j].axis('off')
axes[0,0].set_title('x_t (noisy)', fontsize=10)
axes[0,1].set_title('true x_{t-1}', fontsize=10)
axes[0,2].set_title('pred x_{t-1}', fontsize=10)
plt.suptitle('Naive reverse (one-step): predict x_{t-1} directly', fontsize=12)
plt.tight_layout(); plt.show()

@torch.no_grad()
def sample_naive_xprev(model, n=64, stochastic=False):
    model.eval()
    x = torch.randn(n, 1, 28, 28, device=device)  # start from x_T ~ N(0, I)

    for tt in reversed(range(1, T + 1)):
        t = torch.full((n,), tt, device=device, dtype=torch.long)
        x = model(x, t)

        # Optional: inject noise to mimic a stochastic reverse (very crude).
        if stochastic and tt > 1:
            beta_t = extract(betas, t - 1, x.shape)
            x = x + torch.sqrt(beta_t) * torch.randn_like(x)

    return x.clamp(0, 1)

x_gen_det = sample_naive_xprev(xprev_model, n=100, stochastic=False).cpu().numpy()
x_gen_sto = sample_naive_xprev(xprev_model, n=100, stochastic=True).cpu().numpy()

def show_grid(x_gen, title):
    grid_n = 10
    canvas = np.zeros((28*grid_n, 28*grid_n))
    k = 0
    for i in range(grid_n):
        for j in range(grid_n):
            canvas[i*28:(i+1)*28, j*28:(j+1)*28] = x_gen[k,0]
            k += 1
    plt.figure(figsize=(6.2, 6.2))
    plt.imshow(canvas, cmap='gray')
    plt.axis('off')
    plt.title(title)
    plt.tight_layout(); plt.show()

show_grid(x_gen_det, 'Naive generation (deterministic): repeatedly predict x_{t-1}')
show_grid(x_gen_sto, 'Naive generation (stochastic-ish): add noise during reverse')


---
### Why this direct $x_{t-1}$ prediction tends to struggle

Even if the **one-step** prediction looks reasonable, this approach has fundamental issues:

- **The reverse is stochastic.** Many different $x_{t-1}$ can lead to the same noisy $x_t$ because the forward process adds random noise.
- **MSE encourages averaging.** A deterministic regressor trained with MSE tends to approximate the conditional mean $\mathbb{E}[x_{t-1}\mid x_t]$, which often looks overly smooth (blur).
- **Errors accumulate.** Small one-step errors compound over hundreds of reverse steps, so long rollouts drift off the data manifold.
- **Hard target across all $t$.** The mapping difficulty changes drastically with $t$ (near $t=T$ it is almost pure noise-to-noise).

This motivates a different parameterization of the reverse process.


---
## (5) Reverse denoising, attempt 2: predict the noise $\varepsilon$ (DDPM)

Instead of predicting $x_{t-1}$ directly, DDPMs train a network $\varepsilon_\theta(x_t, t)$ to predict the Gaussian noise used in the forward corruption.

Key identity (closed form):

$$x_t = \sqrt{\bar\alpha_t}\,x_0 + \sqrt{1-\bar\alpha_t}\,\varepsilon, \quad \varepsilon \sim \mathcal{N}(0, I).$$

Why this often works better:

- **Stable target:** for all $t$, the noise $\varepsilon$ has the same simple distribution $\mathcal{N}(0,I)$.
- **Better sampler:** with $\varepsilon_\theta$, we can compute a principled reverse mean and keep sampling stochastic (diverse outputs).

From a predicted noise $\hat\varepsilon = \varepsilon_\theta(x_t,t)$ we can also estimate the clean image:

$$\hat x_0(x_t,t) = \frac{x_t - \sqrt{1-\bar\alpha_t}\,\hat\varepsilon}{\sqrt{\bar\alpha_t}}.$$ 


In [ ]:
# DDPM training objective on MNIST (demo)
# We train eps_theta(x_t, t) to predict the Gaussian noise used to create x_t.

import torch.nn as nn
import torch.nn.functional as F

# Convenience: gather per-t coefficients for a batch of timesteps

def extract(a, t, x_shape):
    # Extract a[t] for each batch index and reshape to (B,1,1,1,...)
    out = a.gather(0, t)
    return out.view(-1, *([1] * (len(x_shape) - 1)))

# Sinusoidal timestep embedding (standard in diffusion / transformers)

def sinusoidal_time_embedding(t, dim=64):
    # t: (B,) int64 -> (B, dim)
    half = dim // 2
    freqs = torch.exp(
        torch.linspace(0, np.log(10000), half, device=t.device, dtype=torch.float32) * (-1)
    )
    args = t.float().unsqueeze(1) * freqs.unsqueeze(0)
    emb = torch.cat([torch.sin(args), torch.cos(args)], dim=1)
    if dim % 2 == 1:
        emb = torch.cat([emb, torch.zeros_like(emb[:, :1])], dim=1)
    return emb

class EpsModel(nn.Module):
    def __init__(self, time_dim=64):
        super().__init__()
        self.time_mlp = nn.Sequential(
            nn.Linear(time_dim, 128), nn.ReLU(),
            nn.Linear(128, 128),
        )
        self.to_time1 = nn.Linear(128, 32)
        self.to_time2 = nn.Linear(128, 64)

        self.conv1 = nn.Conv2d(1, 32, 3, padding=1)
        self.conv2 = nn.Conv2d(32, 64, 3, padding=1)
        self.conv3 = nn.Conv2d(64, 64, 3, padding=1)
        self.conv4 = nn.Conv2d(64, 32, 3, padding=1)
        self.conv5 = nn.Conv2d(32, 1, 3, padding=1)

    def forward(self, x, t):
        t_emb = sinusoidal_time_embedding(t, dim=64)
        t_h = self.time_mlp(t_emb)

        h = F.relu(self.conv1(x) + self.to_time1(t_h).view(-1, 32, 1, 1))
        h = F.relu(self.conv2(h) + self.to_time2(t_h).view(-1, 64, 1, 1))
        h = F.relu(self.conv3(h))
        h = F.relu(self.conv4(h))
        return self.conv5(h)

# Variance schedule for sampling (tilde-beta)
alpha_bar_prev = alpha_bar0[:-1]   # alpha_bar at t-1 (for t=1..T)
alpha_bar_t    = alpha_bar0[1:]    # alpha_bar at t (for t=1..T)

tilde_betas = betas * (1.0 - alpha_bar_prev) / (1.0 - alpha_bar_t)  # length T

def p_sample(model, x_t, t):
    # One reverse step: x_t -> x_{t-1}
    # Here t is in {1,...,T}
    eps = model(x_t, t)

    t_idx = t - 1
    beta_t = extract(betas,  t_idx, x_t.shape)
    alpha_t = extract(alphas, t_idx, x_t.shape)
    a_bar_t = extract(alpha_bar0, t, x_t.shape)

    mean = (1.0 / torch.sqrt(alpha_t)) * (x_t - (beta_t / torch.sqrt(1.0 - a_bar_t)) * eps)

    noise = torch.randn_like(x_t)
    var = extract(tilde_betas, t_idx, x_t.shape)
    nonzero = (t != 1).float().view(-1, 1, 1, 1)
    return mean + nonzero * torch.sqrt(var) * noise

@torch.no_grad()
def sample_ddpm(model, n=64):
    model.eval()
    x = torch.randn(n, 1, 28, 28, device=device)  # start from x_T ~ N(0, I)
    for tt in reversed(range(1, T + 1)):
        t = torch.full((n,), tt, device=device, dtype=torch.long)
        x = p_sample(model, x, t)
    return x.clamp(0, 1)


---
### Reverse sampling with noise prediction (DDPM)

**Training objective** (supervised regression on noise):

1. Sample a clean image $x_0 \sim p_{\text{data}}$.
2. Sample $t \sim \text{Uniform}\{1,\dots,T\}$ and $\varepsilon \sim \mathcal{N}(0,I)$.
3. Form $x_t = \sqrt{\bar\alpha_t}\,x_0 + \sqrt{1-\bar\alpha_t}\,\varepsilon$.
4. Minimize $\lVert \varepsilon_\theta(x_t,t) - \varepsilon \rVert^2$.

**Sampling** (reverse Markov chain): start from $x_T \sim \mathcal{N}(0,I)$ and iterate for $t=T,\dots,1$:

$$\mu_\theta(x_t,t) = \frac{1}{\sqrt{\alpha_t}}\Big(x_t - \frac{\beta_t}{\sqrt{1-\bar\alpha_t}}\,\varepsilon_\theta(x_t,t)\Big),$$

then sample

$$x_{t-1} = \mu_\theta(x_t,t) + \sqrt{\tilde\beta_t}\,z, \quad z \sim \mathcal{N}(0,I),$$

(with no noise added at the final step $t=1$).


---
## (6) Train a tiny diffusion model on MNIST (quick demo)

Below is a minimal DDPM-style training loop.

Notes:
- This is a **teaching demo**, not a high-quality image model.
- Compared to predicting $x_{t-1}$ directly, noise prediction usually trains more stably.
- If it runs slowly, reduce `T`, reduce `epochs`, or train on a subset of MNIST.


In [ ]:
# Train eps_theta

model = EpsModel().to(device)
opt = torch.optim.Adam(model.parameters(), lr=1e-3)

epochs = 3
max_batches_per_epoch = 200  # cap runtime for lecture demo

print('Training noise predictor...')
for epoch in range(epochs):
    model.train()
    total = 0.0

    for batch_idx, (x0, _) in enumerate(train_dl):
        if batch_idx >= max_batches_per_epoch:
            break

        x0 = x0.to(device)
        b = x0.shape[0]

        t = torch.randint(1, T + 1, (b,), device=device, dtype=torch.long)
        eps = torch.randn_like(x0)

        a_bar_t = extract(alpha_bar0, t, x0.shape)
        x_t = torch.sqrt(a_bar_t) * x0 + torch.sqrt(1.0 - a_bar_t) * eps

        eps_pred = model(x_t, t)
        loss = F.mse_loss(eps_pred, eps)

        opt.zero_grad()
        loss.backward()
        opt.step()
        total += loss.item()

    denom = max(1, min(len(train_dl), max_batches_per_epoch))
    print(f"Epoch {epoch+1}/{epochs}  loss={total/denom:.5f}")


In [ ]:
# Generate samples (start from noise -> denoise)

x_gen = sample_ddpm(model, n=100).cpu().numpy()

grid_n = 10
canvas = np.zeros((28*grid_n, 28*grid_n))
k = 0
for i in range(grid_n):
    for j in range(grid_n):
        canvas[i*28:(i+1)*28, j*28:(j+1)*28] = x_gen[k,0]
        k += 1

plt.figure(figsize=(6.2, 6.2))
plt.imshow(canvas, cmap='gray')
plt.axis('off')
plt.title('DDPM demo samples (MNIST)')
plt.tight_layout(); plt.show()


---
## Takeaway

- The forward diffusion is a simple **Gaussian-kernel Markov chain** (data $x_0$ → noise $x_T$).
- A naive reverse model can try to predict $x_{t-1}$ directly, but long rollouts tend to be **blurry/unstable**.
- DDPMs instead predict the noise $\varepsilon$, which gives a more stable training target and a principled stochastic reverse sampler.
